# data_visualization

## Imports and configs

### Fonts setup

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm.findSystemFonts(fontpaths=["/usr/share/fonts/truetype/"])

plt.rcParams["font.family"] = "Times New Roman"

plt.text(0.5, 0.5, 'Teste com Times New Roman', fontsize=12, ha='center')
plt.show()


### Imports

In [1]:
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.manifold import MDS
from skbio.diversity import alpha_diversity
from scipy.spatial.distance import pdist, squareform
from skbio.diversity import beta_diversity

## Data Preparation and Wrangling

In [4]:
target_dir = os.path.expanduser('~/git/sci_init/04_bioreactor_ml_project/data_viz/')
os.chdir(target_dir)
os.getcwd()

'/home/guilherme/git/sci_init/04_bioreactor_ml_project/data_viz'

In [11]:
meta_all = pd.read_csv("../data/metadata/02_metadata_enrichment_clean.csv", index_col="Cluster sample ID")
meta_all.head()

,Metagenomic samples,Cluster sample ID (Larvae/QC_output ),Unnamed: 2,Unnamed: 4,Unnamed: 5,EXPERIMENT,TRANSFER,Gut Compartment,Replicate,Sampling day,...,Acetic_acid_mg/L,Propionic_acid_mg/L,Butyric_acid_mg/L,Cumulative CH4 _mL,Cumulative_CH4 _mL/gVS,H2_%,CO2_%,O2_%,N2_%,CH4_%
Cluster sample ID,,,,,,,,,,,,,,,,,,,,,
UNDR03_2,2,UNDR03_2_1.fastq.tar.gz,UNDR03_2_2.fastq.tar.gz,1st_1T_midA,136,1,1T,Midgut,A,30,...,"4169,61","283,48","140,39","0,61","1,36","0,25","44,05","0,00","50,97","4,73"
UNDR03_3,3,UNDR03_3_1.fastq.tar.gz,UNDR03_3_2.fastq.tar.gz,1st_1T_midB,137,1,1T,Midgut,B,30,...,"4518,43","307,68","80,14","0,80","1,77","0,30","44,22","0,00","50,27","5,21"
UNDR03_4,4,UNDR03_4_1.fastq.tar.gz,UNDR03_4_2.fastq.tar.gz,1st_1T_midC,138,1,1T,Midgut,C,30,...,"4857,67","365,90","110,54","0,18","0,41","0,64","49,82","0,00","48,51","1,03"
UNDR03_5,5,UNDR03_5_1.fastq.tar.gz,UNDR03_5_2.fastq.tar.gz,1st_1T_hidA,139,1,1T,Hindgut,A,30,...,"4133,13","413,10","77,77","3,24","7,20","0,20","47,04","0,00","39,14","13,62"
UNDR03_6,6,UNDR03_6_1.fastq.tar.gz,UNDR03_6_2.fastq.tar.gz,1st_1T_hidB,140,1,1T,Hindgut,B,30,...,"4116,09","379,55","74,02","3,99","8,85","0,00","47,55","0,00","37,30","15,15"


In [12]:
from sklearn.preprocessing import FunctionTransformer
from skbio.stats.composition import multi_replace
from optuna.trial import Trial, FixedTrial
from pandas import DataFrame
def clr_transform(X: np.ndarray | DataFrame) -> np.ndarray | DataFrame:
    """
    Implements the Centered Log Ratio transform, which is recommended for
    dealing with compositional data as such generated by High Throughput
    Sequencing
    """
    # Need to make sure X is at least 2d to not have problems with LeaveOneOut
    X_zeroless = multi_replace(np.asarray(X))
    X_2d = np.atleast_2d(X_zeroless)

    log_X = np.log(X_2d)
    geo_mean = log_X.mean(axis=1, keepdims=True)

    return log_X - geo_mean


class CLRTransformer:
    def create_scaler(self, trial: Trial | FixedTrial):
        return FunctionTransformer(clr_transform, feature_names_out='one-to-one')
asv_all = pd.read_csv("../../04_bioreactor_ml_project/data/genomic/03_map_complete_absolute_n_hits_table.csv", index_col=0).T
asv_all = CLRTransformer().create_scaler(None).set_output(transform="pandas").fit_transform(asv_all)
asv_all

,UNDR01_2HCb-bin.0,UNDR01_2HCb-bin.13,UNDR01_2HCb-bin.15,UNDR01_2HCb-bin.18,UNDR01_2HCb-bin.19,UNDR01_2HCb-bin.21,UNDR01_2HCb-bin.22,UNDR01_2HCb-bin.44,UNDR01_2HCb-bin.49,UNDR01_2HCb-bin.5,...,merge_NG-28520_B96-bin.22,merge_NG-28520_B96-bin.28,merge_NG-28520_B96-bin.29,merge_NG-28520_B96-bin.32,merge_NG-28520_B96-bin.47,merge_NG-28520_B96-bin.48,merge_NG-28520_B96-bin.49,merge_NG-28520_B96-bin.72,merge_NG-28520_B96-bin.8,merge_NG-28520_B96-bin.82
UNDR01_2HCb,5.390542,3.268799,3.924842,4.164478,2.948034,7.259905,3.408711,3.152678,7.525066,4.164121,...,-2.677248,-4.443689,-4.107217,0.742858,-3.507596,-2.635400,2.659674,0.321898,-1.249106,-3.942914
UNDR01_2MCb,4.656991,0.236491,1.300645,1.574035,0.401868,4.408219,0.216123,2.051893,8.294190,1.398128,...,-1.660629,-3.420640,-4.178326,1.530997,-2.792032,-1.756477,3.186812,0.719016,-0.991973,-4.583791
UNDR03_102,-4.126304,-6.118734,-3.012654,-4.578289,-6.118734,-0.844027,-3.783359,-5.271436,-0.591954,-5.831052,...,-0.259849,0.551609,-0.361938,-4.221614,0.923260,1.796614,-0.961597,-2.446662,-2.798506,3.225408
UNDR03_117,-2.462026,-6.666719,-1.661181,-3.894130,-6.666719,-1.554731,-3.055801,-4.181812,-0.326360,-5.973572,...,-3.111371,1.343309,0.472148,-4.101770,1.010218,0.732679,-1.896034,-2.492332,-2.623668,1.604574
UNDR03_118,-2.478680,-6.089598,-6.089598,-2.911544,-6.089598,-1.758864,-3.838306,-6.089598,-0.230237,-4.990985,...,-2.998555,1.278742,0.527137,-5.684133,1.048078,0.806085,-1.658781,-2.426036,-2.705207,2.751417
UNDR03_119,-2.843943,-5.616532,-5.616532,-4.923385,-4.923385,-1.556089,-3.824772,-4.517920,-0.599252,-1.869726,...,-1.538994,0.877222,0.372430,-2.672093,0.954351,0.633443,-2.572009,-2.284327,-2.977475,2.451871
UNDR03_121,-2.906704,-1.784195,-6.690893,-3.746454,-5.997746,-1.628298,-3.746454,-3.800522,-0.304014,-5.997746,...,-0.347013,3.190297,0.601444,-4.493669,0.625655,1.706615,-1.059682,-2.516506,2.256262,3.700683
UNDR03_122,-3.412730,-1.745464,-1.745464,-3.688984,-6.780026,-1.711122,-3.735504,-4.071976,-0.742155,-6.086879,...,0.171746,3.365584,0.883851,-4.477441,0.910717,1.699257,-0.079295,-2.258238,2.953859,1.467980
UNDR03_123,-3.099181,-6.069595,-6.069595,-4.123685,-5.376448,-1.792929,-4.123685,-4.816833,-0.729656,-5.376448,...,0.259234,2.964902,0.737234,-4.683301,0.682675,1.525540,-0.164234,-2.573088,2.819989,2.626664
UNDR03_1HAb,5.880708,2.154382,1.618177,3.224087,-1.086358,0.423550,3.607160,0.608774,3.869654,-0.596810,...,-4.030797,-1.833429,-2.239038,-0.258036,-2.932185,-3.337650,-3.625332,-0.269597,-3.625332,-2.932185


In [13]:
merged = pd.merge(meta_all[['Gut Compartment', 'EXPERIMENT', 'TRANSFER']], asv_all,
                  how='inner', left_index=True, right_index=True)
merged.info()

<class 'pandas.DataFrame'>
Index: 24 entries, UNDR03_2 to UNDR03_123
Columns: 592 entries, Gut Compartment to merge_NG-28520_B96-bin.82
dtypes: float64(589), int64(1), str(2)
memory usage: 111.6 KB


In [14]:
meta_all.index

Index(['UNDR03_2', 'UNDR03_3', 'UNDR03_4', 'UNDR03_5', 'UNDR03_6', 'UNDR03_7',
       'UNDR03_22', 'UNDR03_23', 'UNDR03_24', 'UNDR03_25', 'UNDR03_26',
       'UNDR03_27', 'UNDR03_74', 'UNDR03_75', 'UNDR03_76', 'UNDR03_78',
       'UNDR03_79', 'UNDR03_80', 'UNDR03_117', 'UNDR03_118', 'UNDR03_119',
       'UNDR03_121', 'UNDR03_122', 'UNDR03_123'],
      dtype='str', name='Cluster sample ID')

In [18]:
def update_categoria(row):
    return f"{row['TRANSFER']}-{row['EXPERIMENT']}"
meta_all["TRANSFER"] = meta_all.apply(update_categoria, axis=1)
meta_all["TRANSFER"].unique()

<ArrowStringArray>
['1T-1', '3T-1', '1T-2', '3T-2']
Length: 4, dtype: str

## Visual Analysis

### Relevant imports

In [25]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.manifold import MDS
from skbio.diversity import alpha_diversity
from skbio.stats.distance import permanova
from scipy.spatial.distance import squareform
from scipy.spatial import distance_matrix
from scipy.stats import ttest_ind
from skbio import DistanceMatrix
from scipy.stats import kruskal

In [21]:
metadata = meta_all.loc[:, ['Gut Compartment', 'EXPERIMENT', 'TRANSFER']].copy()
metadata.shape

(24, 3)

### Markers for Plots

In [38]:
px_markers = {
    '3T-1': 2,         # diamond
    '3T-2': 302,       # open diamond
    '1T-1': 0,         # círculo
    '1T-2': 300          # circulo aberto com ponto
}

### Distance

In [35]:
asv_all = asv_all.loc[meta_all.index]
dist_matrix = pdist(asv_all)
dist_matrix_square = squareform(dist_matrix)
print(dist_matrix.shape)
print(dist_matrix_square.shape)

(276,)
(24, 24)


### NMDS

In [41]:
order = ['1T-1', '1T-2','3T-1', '3T-2']
mds = MDS(n_components=2, metric='precomputed', random_state=42,
          metric_mds=True, n_init=4, init='random')
nmds_results = mds.fit_transform(dist_matrix_square)

metadata['NMDS1'] = nmds_results[:, 0]
metadata['NMDS2'] = nmds_results[:, 1]
import plotly.express as px
fig = px.scatter(metadata, x='NMDS1', y='NMDS2', color='Gut Compartment',
                symbol='TRANSFER', category_orders={'Gut Compartment': ['Midgut', 'Hindgut'],
                                                    'Category': order},
                symbol_map=px_markers, title='nMDS of Microbial Distribution')
fig.update_traces(marker={'size':14})
fig.update_layout(width=1200, height=700)
y_range = [metadata['NMDS2'].min()-0.1, metadata['NMDS2'].max()+0.1]
x_range = [metadata['NMDS1'].min()-0.1, metadata['NMDS1'].max()+0.1]
fig.update_yaxes(range=y_range)
fig.update_xaxes(range=x_range)
for point in fig.data:
    group = point.legendgroup.split(',')[0]
    point.legendgroup = group
    point.legendgrouptitle.text = group
    count = len(point.x)
    point.name = point.name.split(',')[1].strip() + f' (n = {count})'
fig.update_layout(legend_groupclick="toggleitem")

show_all = [True] * len(fig.data)
hide_all = ['legendonly'] * len(fig.data)
midgut_only = [True if trace.legendgroup == 'Midgut' else 'legendonly' for trace in fig.data]
hindgut_only = [True if trace.legendgroup == 'Hindgut' else 'legendonly' for trace in fig.data]

fig.update_layout(
    title_pad_l=300,
    updatemenus=[
        dict(
            type='buttons',
            direction='left',
            xanchor='left',
            x=1.0, # Adjust X and Y to position the buttons where you like
            y=1.16,
            showactive=False,
            buttons=[
                dict(label='Show All', method='restyle', args=['visible', show_all]),
                dict(label='Hide All', method='restyle', args=['visible', hide_all]),
            ]
        ),
        dict(
            type='buttons',
            direction='left',
            xanchor='left',
            x=1.0, # Adjust X and Y to position the buttons where you like
            y=1.1,
            showactive=False,
            buttons=[
                dict(label='Midgut Only', method='restyle', args=['visible', midgut_only]),
                dict(label='Hindgut Only', method='restyle', args=['visible', hindgut_only]),
            ]
        )
    ]
)
fig.show()
fig.write_html('nmds_gut_cat_interactive.html')

### PCA


In [47]:
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. Prepare your data
# Note: Replace 'feature_matrix' with your actual sample-by-feature abundance dataframe.
# Standardizing the features is highly recommended for PCA.

# 2. Run PCA
pca = PCA(n_components=2, random_state=42)
pca_results = pca.fit_transform(asv_all)

# Extract explained variance to make your axis labels highly informative
exp_var = pca.explained_variance_ratio_ * 100

# Add PCA results to your metadata dataframe
metadata['PC1'] = pca_results[:, 0]
metadata['PC2'] = pca_results[:, 1]

# 3. Plotting with Plotly Express
order = ['1T-1', '1T-2', '3T-1', '3T-2']

fig = px.scatter(
    metadata, 
    x='PC1', 
    y='PC2', 
    color='Gut Compartment',
    symbol='TRANSFER', 
    category_orders={'Gut Compartment': ['Midgut', 'Hindgut'], 'Category': order},
    symbol_map=px_markers, 
    title='PCA of CLR-Transformed Microbial Distribution',
    labels={'PC1': f'PC1 ({exp_var[0]:.1f}%)', 'PC2': f'PC2 ({exp_var[1]:.1f}%)'}
)

# 4. Styling and Layout Customization (Preserving your exact setup)
fig.update_traces(marker={'size':14})
fig.update_layout(width=1200, height=700)

y_range = [metadata['PC2'].min() - 0.1, metadata['PC2'].max() + 0.1]
x_range = [metadata['PC1'].min() - 0.1, metadata['PC1'].max() + 0.1]
fig.update_yaxes(range=y_range)
fig.update_xaxes(range=x_range)

# Clean up legend groups and names
for point in fig.data:
    group = point.legendgroup.split(',')[0]
    point.legendgroup = group
    point.legendgrouptitle.text = group
    count = len(point.x)
    point.name = point.name.split(',')[1].strip() + f' (n = {count})'

fig.update_layout(legend_groupclick="toggleitem")

# Define visibility arrays for the interactive buttons
show_all = [True] * len(fig.data)
hide_all = ['legendonly'] * len(fig.data)
midgut_only = [True if trace.legendgroup == 'Midgut' else 'legendonly' for trace in fig.data]
hindgut_only = [True if trace.legendgroup == 'Hindgut' else 'legendonly' for trace in fig.data]

# Add custom interactive buttons
fig.update_layout(
    title_pad_l=300,
    updatemenus=[
        dict(
            type='buttons',
            direction='left',
            xanchor='left',
            x=1.0, 
            y=1.16,
            showactive=False,
            buttons=[
                dict(label='Show All', method='restyle', args=['visible', show_all]),
                dict(label='Hide All', method='restyle', args=['visible', hide_all]),
            ]
        ),
        dict(
            type='buttons',
            direction='left',
            xanchor='left',
            x=1.0, 
            y=1.1,
            showactive=False,
            buttons=[
                dict(label='Midgut Only', method='restyle', args=['visible', midgut_only]),
                dict(label='Hindgut Only', method='restyle', args=['visible', hindgut_only]),
            ]
        )
    ]
)

# Show and save the interactive plot
fig.show()
fig.write_html('pca_gut_cat_interactive.html')